# IB9LQ0 – Generative AI and AI Applications
## Required Task 9 – OCR with Gemini VLM: Receipt Total Price Extraction

**Student Name:** Jashwanth Anand Shankar  
**Student Number:** *(add your student number here)*  
**Module:** IB9LQ0 | Warwick Business School | 2025–2026

---

### Key Insights and Takeaways

**Task Overview**

This task uses the Gemini 2.5 Flash Vision-Language Model to extract the `total_price` from receipt images in the `naver-clova-ix/cord-v2` dataset. Predictions are evaluated against the ground-truth values in `df_receipt`.

**Key Insight 1 – Prompt design controls output format**

Asking the model to return only a float — with no currency symbol or label — simplifies post-processing. The prompt acts as the output specification for the VLM.

**Key Insight 2 – Error handling is essential**

Real receipts produce noisy responses. Wrapping each API call in a try/except block and using regex to extract numeric values ensures the pipeline records NaN gracefully without crashing.

**Key Insight 3 – Three metrics give a complete picture**

MAE measures average monetary error. Successful extraction rate measures how often the model returned a valid number. Accuracy within 5% and 10% captures near-correct predictions that MAE would penalise too harshly.

**Personal Takeaway**

Running the Gemini API in a loop over 15 receipt images showed that the bottleneck is network latency, not model compute. A short sleep between calls prevents rate-limit errors and reflects how a real production pipeline would be managed.

---
*AI Disclosure: Claude (Anthropic) was used to assist in structuring and drafting this code. All outputs, interpretations, and conclusions were reviewed and validated by the student.*

# Setup

The cells below follow the environment setup from the lecture notebook (`MLM2_OCR_with_Gemini.ipynb`). They load the Gemini API key from Colab Secrets, list available models, and confirm the model to be used.

In [ ]:
from google.colab import userdata
import os

# Load the API key from Colab Secrets and set it as an environment variable.
# The secret must be named 'GEMINI_API_KEY' in the Colab Secrets sidebar.
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("API key loaded and environment variable set.")

In [ ]:
from google import genai

# Initialise the Gemini client — reads GEMINI_API_KEY from the environment.
client = genai.Client()

# List available Gemini models (reproduced from the lecture notebook).
print("Listing available Gemini models:")
for m in client.models.list():
    if "gemini" in m.name.lower():
        print(f"Name: {m.name}, Description: {m.description}")

In [ ]:
# Set the model to use throughout this task, consistent with the lecture.
model_id = 'gemini-2.5-flash'

print(f"Set to use Google Gemini model: {model_id}")

# Reconstruct `df_receipt` from the CORD-v2 Dataset

Before starting the required task we rebuild the `df_receipt` DataFrame exactly as done in the lecture notebook. This gives us the ground-truth `total_price` column needed for evaluation.

Steps taken from the lecture:
1. Load `naver-clova-ix/cord-v2` via the `datasets` library.
2. Parse each record's JSON `ground_truth` to extract `image` and `total_price`.
3. Drop the `store_name` column (found to be entirely `None` in the lecture).

In [ ]:
from datasets import load_dataset

# Load the CORD-v2 receipt dataset — same dataset used in the MLM2 lecture.
ds = load_dataset("naver-clova-ix/cord-v2")

print("Dataset loaded successfully.")
print(ds)

In [ ]:
import pandas as pd
import json
import re

# Reproduce the exact extraction loop from the lecture notebook.
# Iterate through ds['train'], parse the ground_truth JSON,
# and extract the PIL image plus the total_price float.

receipt_data_df = []

for record in ds['train']:
    image            = record['image']
    ground_truth_str = record['ground_truth']

    store_name  = None
    total_price = None

    try:
        ground_truth_json = json.loads(ground_truth_str)
        gt_parse = ground_truth_json.get('gt_parse', {})

        # Store name extraction (kept for parity with lecture; dropped later).
        company_name = gt_parse.get('company', {}).get('name')
        seller_name  = gt_parse.get('seller',  {}).get('name')
        if company_name:
            store_name = company_name
        elif seller_name:
            store_name = seller_name

        # Total price extraction.
        total_info      = gt_parse.get('total', {})
        total_price_str = total_info.get('total_price')

        # Clean and convert the price string to float.
        # Handles commas-as-decimal, multiple dots, and currency symbols.
        if total_price_str is not None and isinstance(total_price_str, str):
            cleaned_price = total_price_str.replace(',', '.')

            parts = re.findall(r'\d+', cleaned_price)
            if parts:
                last_dot_index = cleaned_price.rfind('.')
                if last_dot_index != -1 and last_dot_index > cleaned_price.rfind(parts[-1]):
                    integer_part = ''.join(re.findall(r'\d', cleaned_price[:last_dot_index]))
                    decimal_part = ''.join(re.findall(r'\d', cleaned_price[last_dot_index + 1:]))
                    cleaned_price = f"{integer_part}.{decimal_part}"
                else:
                    cleaned_price = ''.join(parts)
            else:
                cleaned_price = ""

            try:
                total_price = float(cleaned_price) if cleaned_price else None
            except ValueError:
                total_price = None

        receipt_data_df.append({
            'image':       image,
            'store_name':  store_name,
            'total_price': total_price
        })

    except json.JSONDecodeError as e:
        print(f"JSON decode error for a record: {e}")
        continue
    except Exception as e:
        print(f"Unexpected error for a record: {e}")
        continue

df_receipt_summary = pd.DataFrame(receipt_data_df)

print(f"Created df_receipt_summary with {len(df_receipt_summary)} records.")
print("First 5 rows of df_receipt_summary:")
display(df_receipt_summary.head())

In [ ]:
# Assign to df_receipt as per the lecture notebook.
df_receipt = df_receipt_summary

print(f"Number of records in df_receipt: {len(df_receipt)}")
print("First 5 rows of df_receipt:")
display(df_receipt.head())

In [ ]:
# Check how many store_name values are None (reproduced from the lecture).
num_none_store_name = df_receipt['store_name'].isnull().sum()
print(f"Number of records with None in 'store_name': {num_none_store_name}")

In [ ]:
# Drop the store_name column as it was found to be entirely None in the lecture.
df_receipt = df_receipt.drop(columns=['store_name'])
print(" 'store_name' column removed from df_receipt.")
print("First 5 rows of df_receipt after column removal:")
display(df_receipt.head())

# Required Task 9

The five steps below correspond directly to the task specification.

## Step 1: Randomly Select 15 Records

We randomly sample 15 receipts from `df_receipt` to form our test set.
A fixed `random_state` is used so the selection is reproducible across notebook runs.
Only rows where `total_price` is not `NaN` are included in the sampling pool,
ensuring every selected receipt has a valid ground-truth value for evaluation.

**Note on the task wording:** The task description contains a typographical inconsistency — the instruction says 'randomly select 100 receipts' but the step heading and DataFrame name both specify 15. This notebook uses n=15 as the correct interpretation, consistent with the heading 'Randomly Select 15 Records' and the DataFrame name `df_test_receipts`.

In [ ]:
import pandas as pd

# Filter out rows where total_price is NaN before sampling.
# This guarantees every sampled receipt has a valid ground-truth price.
df_receipt_valid = df_receipt.dropna(subset=['total_price']).reset_index(drop=True)

print(f"Total records in df_receipt          : {len(df_receipt)}")
print(f"Records with valid total_price        : {len(df_receipt_valid)}")

# Randomly select 15 receipts. random_state=42 ensures reproducibility.
df_test_receipts = df_receipt_valid.sample(n=15, random_state=42).reset_index(drop=True)

print(f"\nRandomly selected {len(df_test_receipts)} receipts for testing.")
print("\ndf_test_receipts — total_price (ground truth):")
display(df_test_receipts[['total_price']].head())

## Step 2: Define a Prompt for Gemini

The prompt instructs Gemini to extract only the grand total from the receipt image
and return it as a single float — no currency symbol, no label, no explanation.
Explicit constraints minimise the chance of the model returning extra text
that would break float conversion in Step 3.

In [ ]:
# The prompt instructs Gemini to return ONLY the final total amount as a float.
extraction_prompt = (
    "Look at this receipt image carefully. "
    "Extract the final total amount — the grand total that the customer paid. "
    "Provide ONLY the numerical value as a float (e.g., 12.50). "
    "Do NOT include any currency symbol, label, explanation, or additional text. "
    "Return a single numerical value only."
)

print("Extraction prompt defined:")
print("-" * 60)
print(extraction_prompt)
print("-" * 60)

## Step 3: Process `df_test_receipts` with Gemini

We iterate through each row of `df_test_receipts`, pass the receipt image
and the prompt to the Gemini VLM, and parse the response into a numeric value.
Each API call is wrapped in a try/except block. If no valid float can be extracted,
`float('nan')` is assigned so the loop continues without crashing.

In [ ]:
import re
import time

predicted_prices = []
total_receipts = len(df_test_receipts)

print(f"Starting Gemini predictions for {total_receipts} receipts...")
print("=" * 60)

for idx, row in df_test_receipts.iterrows():
    receipt_image = row['image']
    ground_truth  = row['total_price']

    predicted_price = float('nan')

    try:
        # Call the Gemini VLM with the prompt and the PIL image.
        response = client.models.generate_content(
            model=model_id,
            contents=[extraction_prompt, receipt_image]
        )

        response_text = response.text.strip()

        # Parse the response to extract a float.
        # Replace comma-as-decimal, then extract numeric substrings.
        cleaned_response = response_text.replace(',', '.')
        numeric_matches = re.findall(r'\d+\.?\d*', cleaned_response)

        if numeric_matches:
            predicted_price = float(numeric_matches[0])
        else:
            print(f"  [Row {idx}] Warning: No numeric value found in response: '{response_text}'")
            predicted_price = float('nan')

    except Exception as e:
        print(f"  [Row {idx}] API error: {e}")
        predicted_price = float('nan')

    predicted_prices.append(predicted_price)

    print(f"  Receipt {len(predicted_prices):>2}/{total_receipts} | "
          f"Ground Truth: {ground_truth:>8.2f} | "
          f"Predicted: {predicted_price if not pd.isna(predicted_price) else 'NaN':>8}")

    # Short sleep to avoid hitting Gemini free-tier rate limits.
    time.sleep(2)

# Add the predictions as a new column to df_test_receipts.
df_test_receipts = df_test_receipts.copy()
df_test_receipts['predicted_total_price'] = predicted_prices

print("=" * 60)
print("Gemini predictions complete. 'predicted_total_price' column added.")

## Step 4: Evaluate Predictions

We compute three evaluation metrics as specified in the task:
- **Successful extractions**: how many predictions are valid (non-NaN)
- **MAE**: average absolute monetary error over valid prediction pairs
- **Accuracy within 5% and 10%**: percentage of predictions within each threshold

All metrics are computed only over rows where both `total_price` and
`predicted_total_price` are non-NaN.

In [ ]:
# Successful extractions.
num_successful = df_test_receipts['predicted_total_price'].notna().sum()
num_total      = len(df_test_receipts)
extraction_rate = (num_successful / num_total) * 100

# Build evaluation DataFrame with valid prediction pairs only.
df_eval = df_test_receipts.dropna(subset=['total_price', 'predicted_total_price']).copy()

# Mean Absolute Error (MAE).
df_eval['absolute_error'] = (df_eval['predicted_total_price'] - df_eval['total_price']).abs()
mae = df_eval['absolute_error'].mean()

# Accuracy within a threshold.
# Relative error = |predicted - actual| / |actual| * 100.
# Guard against division by zero for any ground-truth price of 0.
df_eval['relative_error_pct'] = (
    df_eval['absolute_error'] /
    df_eval['total_price'].replace(0, float('nan'))
) * 100

num_within_5pct  = (df_eval['relative_error_pct'] <= 5).sum()
num_within_10pct = (df_eval['relative_error_pct'] <= 10).sum()

accuracy_5pct  = (num_within_5pct  / len(df_eval)) * 100 if len(df_eval) > 0 else 0.0
accuracy_10pct = (num_within_10pct / len(df_eval)) * 100 if len(df_eval) > 0 else 0.0

print("Evaluation metrics calculated.")
print(f"  Valid prediction pairs for metric calculation: {len(df_eval)}")

## Step 5: Display Results

We print the evaluation metrics in a formatted summary, then display
`df_test_receipts` showing `total_price` and `predicted_total_price`
side by side for comparison.

In [ ]:
print("=" * 60)
print("       GEMINI VLM — RECEIPT TOTAL PRICE EXTRACTION")
print("                  EVALUATION RESULTS")
print("=" * 60)
print(f"  Total receipts tested             : {num_total}")
print(f"  Successful extractions            : {num_successful} / {num_total} "
      f"({extraction_rate:.1f}%)")
print("-" * 60)
print(f"  Mean Absolute Error (MAE)         : {mae:.4f}")
print(f"  Accuracy within  5% of truth      : {num_within_5pct} / {len(df_eval)} "
      f"({accuracy_5pct:.1f}%)")
print(f"  Accuracy within 10% of truth      : {num_within_10pct} / {len(df_eval)} "
      f"({accuracy_10pct:.1f}%)")
print("=" * 60)

In [ ]:
# Display all 15 rows showing ground truth vs predicted with error columns.
df_display = df_test_receipts[['total_price', 'predicted_total_price']].copy()
df_display['absolute_error'] = (
    df_display['predicted_total_price'] - df_display['total_price']
).abs()
df_display['relative_error_%'] = (
    df_display['absolute_error'] /
    df_display['total_price'].replace(0, float('nan'))
) * 100

df_display = df_display.round({
    'total_price':           2,
    'predicted_total_price': 2,
    'absolute_error':        2,
    'relative_error_%':      2
})

print("df_test_receipts — Ground Truth vs. Predicted Total Price:")
display(df_display)